# EXERCISE 1 - Gradient descent & Deep Neural Networks

Group 2607
* Daniele Ghezzi ... (matricola)
* Fabio Cimino ...
* Riccardo Ferrante ...
* Federico Scianna ... 

## 1. Random search of best hyperparameters

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras_tuner import RandomSearch
from sklearn.preprocessing import StandardScaler
import tensorflow.random as tf_r

%run useful.py

plt.rcParams['font.size'] = 13

N=12000
L=8
B=10

TYPE=4

# we set the seeds for reproducibility
np.random.seed(12345)
tf_r.set_seed(12345)

In [20]:
# preprocessing
def Standardize(x,m,s):
    """
    rescale each component using its mean and standard deviation
    x: data points (numpy array)
    m: mean values (numpy array)
    s: standard deviations (numpy array)
    return: rescaled data points (numpy array)
    """
    N = len(x)
    # assuming len(m)=len(s)=len(x[0])
    mm,ss = np.tile(m,(N,1)), np.tile(s,(N,1))
    return (x-mm)/ss

perc_train = 0.7
perc_valid = 0.15
perc_test = 0.15

x = np.loadtxt(filename("data",L,TYPE), delimiter=' ')
y = np.loadtxt(filename("labels",L,TYPE), delimiter=' ', dtype=int)

N_train = int(perc_train * N)
N_val = int(perc_valid * N)
N_test = N - N_train - N_val
print(f'data: {N}\ntrain: {N_train}\nvalid: {N_val}\ntest: {N_test}')

# split data into train/validation/test sets and check 
(x_train, y_train) = (x[0:N_train],y[0:N_train])
(x_val, y_val) = (x[N_train:N_train+N_val],y[N_train:N_train+N_val])
(x_test, y_test) = (x[N_train+N_val:],y[N_train+N_val:])
print("Train:",len(x_train),"\t Validation:",len(x_val),"\t Test:",len(x_test))

x_train_mean = np.mean(x_train, axis=0)
x_train_std = np.std(x_train, axis=0)
x_val_mean = np.mean(x_val, axis=0)
x_val_std = np.std(x_val, axis=0)
x_test_mean = np.mean(x_test, axis=0)
x_test_std = np.std(x_test, axis=0)
x_train = Standardize(x_train, x_train_mean, x_train_std)
x_val = Standardize(x_val, x_val_mean, x_val_std)
x_test = Standardize(x_test, x_test_mean, x_test_std)
print("after rescaling (train):\nmean value=", x_train.mean(axis=0), "\nstd. dev.=", x_train.std(axis=0))

data: 12000
train: 8400
valid: 1800
test: 1800
Train: 8400 	 Validation: 1800 	 Test: 1800
after rescaling (train):
mean value= [ 9.17572896e-16  1.18834043e-14 -1.95478554e-15 -3.53337069e-15
 -6.42594443e-15 -2.40365928e-15 -2.88332932e-15 -8.27623684e-15] 
std. dev.= [1. 1. 1. 1. 1. 1. 1. 1.]


In [ ]:
import keras_tuner as kt
from tensorflow import keras

def build_model(hp):
    
    model = keras.Sequential()
    
    # we define the activation and dropout hyperparameters to tune
    hp_activation = hp.Choice('activation', values=['relu', 'tanh', 'sigmoid', 'elu'])
    hp_dropout_rate = hp.Choice('dropout_rate', values=[0.0, 0.1, 0.2])
    
    # we build the architecture (3 layers, 20 neurons each)
    model.add(keras.layers.Dense(20, activation=hp_activation, input_shape=(L,)))
    model.add(keras.layers.Dropout(rate=hp_dropout_rate))

    model.add(keras.layers.Dense(20, activation=hp_activation))
    model.add(keras.layers.Dropout(rate=hp_dropout_rate))

    model.add(keras.layers.Dense(20, activation=hp_activation))
    model.add(keras.layers.Dropout(rate=hp_dropout_rate))

    model.add(keras.layers.Dense(1, activation='sigmoid'))
    
    # we set the optimizer and learning rate hyperparameters to tune
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6])
    hp_optimizer = hp.Choice('optimizer', values=['adam', 'rmsprop', 'sgd'])
    
    if hp_optimizer == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=hp_learning_rate)
    elif hp_optimizer == 'rmsprop':
        optimizer = keras.optimizers.RMSprop(learning_rate=hp_learning_rate)
    else:
        optimizer = keras.optimizers.SGD(learning_rate=hp_learning_rate, nesterov=True)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [25]:
# 1. Initialize the RandomSearch tuner
tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective='val_accuracy',
    max_trials=20,  # The maximum number of random combinations to test
    executions_per_trial=1, # Number of models to train per trial (increase to reduce variance)
    directory='my_tuning_dir',
    project_name='random_search_demo'
)

# 2. View a summary of the search space
tuner.search_space_summary()

# 3. Execute the search (assuming you have x_train, y_train, x_val, y_val)
tuner.search(
    x_train, 
    y_train, 
    epochs=10, 
    validation_data=(x_val, y_val)
)

# 4. Retrieve the best results
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best Activation Function: {best_hps.get('activation')}")
print(f"Best Optimizer: {best_hps.get('optimizer')}")
print(f"Best Learning Rate: {best_hps.get('learning_rate')}")
print(f"Best Dropout Rate: {best_hps.get('dropout_rate')}")

# Get the actual best model to use for predictions
best_model = tuner.get_best_models()[0]

Reloading Tuner from my_tuning_dir/random_search_demo/tuner0.json
Search space summary
Default search space size: 4
activation (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'tanh', 'sigmoid', 'elu'], 'ordered': False}
dropout_rate (Choice)
{'default': 0.0, 'conditions': [], 'values': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5], 'ordered': True}
learning_rate (Choice)
{'default': 0.1, 'conditions': [], 'values': [0.1, 0.01, 0.001, 0.0001, 1e-05, 1e-06], 'ordered': True}
optimizer (Choice)
{'default': 'adam', 'conditions': [], 'values': ['adam', 'rmsprop', 'sgd'], 'ordered': False}
Best Activation Function: sigmoid
Best Optimizer: rmsprop
Best Learning Rate: 0.1
Best Dropout Rate: 0.1


/home/dghezzi/LaboratoryOfComputationalPhysics_Y8/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/dghezzi/LaboratoryOfComputationalPhysics_Y8/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
